# RETINA-AI · Kişi 4: Derin Öğrenme (EfficientNet-B0 + Grad-CAM)

**Çalıştırmadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`

Akış:
1. Drive bağlantısı + kodu GitHub'dan çekme
2. Veriyi Drive'dan Colab'ın yerel diskine kopyalama (Drive'dan doğrudan okumak eğitimi ~10 kat yavaşlatır)
3. Hızlı deneme (200 görüntü, 2 epoch) → her şey çalışıyor mu?
4. Tam eğitim (erken durdurmalı)
5. Test tahminleri (Kişi 5 için CSV) + Grad-CAM figürü
6. Çıktıları Drive'a kaydetme

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Kodu GitHub'dan cek: once K4 branch'i, o yoksa (merge edildiyse) main
!test -d /content/retina-ai || git clone -q https://github.com/blosny/retina-ai-ocular-disease-classification.git /content/retina-ai
%cd /content/retina-ai
!git fetch -q origin feature/derin-ogrenme-cnn && git checkout -q -B feature/derin-ogrenme-cnn FETCH_HEAD || (git checkout -q main && git pull -q origin main)

PROJE = Path('/content/retina-ai')
DRIVE = Path('/content/drive/MyDrive/retina-ai-ocular-disease-classification')
sys.path.insert(0, str(PROJE))

if not (PROJE / 'src' / 'cnn_models.py').exists():
    raise FileNotFoundError("src/cnn_models.py bulunamadi: feature/derin-ogrenme-cnn branch'i GitHub'a push edilmis mi kontrol edin.")
if not DRIVE.exists():
    raise FileNotFoundError("Drive klasoru bulunamadi: paylasilan klasoru 'Drive'a kisayol ekle -> Drive'im' ile ekleyin.")
print('Kod ve Drive hazir.')

## 2. Veriyi yerel diske kopyala
Görüntülerin `<sınıf>/<dosya>.jpg` yapısında olduğu klasörü bulur (`data/raw`, `archive/dataset` veya `dataset`). Zip olarak durduysa açar.

In [ ]:
import shutil, zipfile
YEREL = Path('/content/dataset')

if not (YEREL / 'glaucoma').is_dir():
    adaylar = [DRIVE/'data'/'raw', DRIVE/'archive'/'dataset', DRIVE/'dataset', DRIVE/'data'/'raw'/'dataset']
    kaynak = next((a for a in adaylar if (a/'glaucoma').is_dir()), None)
    zipler = list(DRIVE.rglob('*.zip'))
    if kaynak:
        print('Kopyalaniyor:', kaynak); shutil.copytree(kaynak, YEREL)
    elif zipler:
        print('Zip aciliyor:', zipler[0])
        zipfile.ZipFile(zipler[0]).extractall('/content/_zip')
        ic = next(p.parent for p in Path('/content/_zip').rglob('glaucoma') if p.is_dir())
        shutil.move(str(ic), str(YEREL))
    else:
        raise FileNotFoundError('Drive klasorunde goruntuler bulunamadi, klasor yapisini kontrol edin.')

for s in ['normal', 'diabetic_retinopathy', 'glaucoma', 'cataract']:
    print(s, len(list((YEREL/s).glob('*'))))

## 3. Hızlı deneme (2-3 dakika)
Hata çıkmazsa tam eğitime geç.

In [ ]:
from src import cnn_models as cnn

cnn.egit(PROJE, veri_koku=YEREL, model_yolu=PROJE/'models'/'deneme.pth',
         epoch=2, ornek_sayisi=200)

## 4. Tam eğitim
- 384×384 girdi, batch 16, AdamW (lr=3e-4), en fazla 30 epoch
- Erken durdurma: doğrulama kaybı 5 epoch iyileşmezse durur
- En iyi model doğrudan Drive'a kaydedilir (Colab kapansa da kaybolmaz)
- T4'te epoch başına ~1 dk beklenir. Yavaşsa `boyut=224` yapın.

In [ ]:
# Ortak klasorde yazma izniniz yoksa kendi Drive'iniza kaydedin
KAYIT = Path('/content/drive/MyDrive/retina-ai-k4')
MODEL_YOLU = KAYIT / 'models' / 'model_efficientnet.pth'

gecmis = cnn.egit(PROJE, veri_koku=YEREL, model_yolu=MODEL_YOLU,
                  epoch=30, sabir=5, batch=16, boyut=384, girdi_modu='rgb')

from IPython.display import Image, display
display(Image(str(PROJE/'reports'/'figures'/'cnn_egitim_grafigi.png')))

## 5. Test tahminleri + Grad-CAM
- `reports/cnn_test_tahminleri.csv`: her test görüntüsü için gerçek sınıf, tahmin ve 4 olasılık (Kişi 5 metrikleri buradan hesaplar)
- `reports/figures/gradcam_ornekleri.png`: her sınıftan 1 doğru + 1 yanlış tahmin

**Grad-CAM'i yorumlarken:** Isı optik disk (glokom), lezyonlar (DR) veya tüm retina (katarakt) üzerinde mi? Siyah kenarlarda veya köşelerdeyse model kamerayı/kaynağı öğrenmiş olabilir.

In [ ]:
tahminler = cnn.test_tahminlerini_kaydet(PROJE, veri_koku=YEREL, model_yolu=MODEL_YOLU)
print(tahminler.groupby('gercek')['dogru'].mean().round(3))

cnn.gradcam_figuru_uret(tahminler, PROJE, veri_koku=YEREL, model_yolu=MODEL_YOLU)
display(Image(str(PROJE/'reports'/'figures'/'gradcam_ornekleri.png')))

## 5b. Kenar testi (kaynak/kamera kisayolu kontrolu)
Test goruntulerinin dis halkasi siyahla kapatilip model tekrar calistirilir.
- Kenar kapatilinca bir sinifin dogrulugu digerlerinden cok daha fazla dusuyorsa model o sinifi kenar/kamera izlerinden taniyor olabilir.
- `sadece_kenar` kosulunda retinanin ortasi tamamen siyahtir; bu goruntu "detay yok" oldugu icin katarakta benzer, tek basina kisayol kaniti degildir.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn.functional as F

model = cnn.model_yukle(MODEL_YOLU)
donusum = cnn.donusum_olustur(model.bilgi['goruntu_boyutu'], egitim=False)
cihaz = next(model.parameters()).device
df = pd.read_csv(PROJE/'data'/'splits'/'veri_bolme.csv')
df = df[df['split'] == 'test'].reset_index(drop=True)

yy, xx = np.mgrid[:512, :512]
uzaklik = np.sqrt((xx - 256)**2 + (yy - 256)**2) / 256   # 0 = merkez, 1 = kenar
KOSULLAR = {                                              # True olan pikseller KALIR
    'orijinal':          np.ones((512, 512), bool),
    'kenar_kapali_%15':  uzaklik <= 0.85,
    'kenar_kapali_%30':  uzaklik <= 0.70,
    'sadece_kenar':      uzaklik > 0.85,
}

satirlar = []
with torch.no_grad():
    for _, s in df.iterrows():
        rgb = cnn.cnn_girdisi_hazirla(YEREL / s['relative_path'])
        x = torch.stack([donusum(rgb * m[..., None].astype(np.uint8)) for m in KOSULLAR.values()]).to(cihaz)
        tahminler_k = F.softmax(model(x), 1).argmax(1).cpu().numpy()
        for ad, t in zip(KOSULLAR, tahminler_k):
            satirlar.append({'gercek': s['class_name'], 'kosul': ad, 'tahmin': cnn.SINIFLAR[t]})

sonuc = pd.DataFrame(satirlar)
sonuc['dogru'] = sonuc['gercek'] == sonuc['tahmin']

print('1) SINIF BAZINDA DOGRULUK (%)')
tablo = (sonuc.pivot_table(index='gercek', columns='kosul', values='dogru', aggfunc='mean') * 100).round(1)
display(tablo[list(KOSULLAR)])

print('\n2) SADECE KENAR VERILINCE "KATARAKT" DENEN ORAN (%)')
kenar = sonuc[sonuc['kosul'] == 'sadece_kenar']
display((kenar.assign(katarakt=kenar['tahmin'] == 'cataract')
             .groupby('gercek')['katarakt'].mean() * 100).round(1))

sonuc.to_csv(PROJE/'reports'/'cnn_kenar_testi.csv', index=False)
tablo.to_csv(PROJE/'reports'/'cnn_kenar_testi_ozet.csv')

## Tek görüntü denemesi (arayüzdeki çağrının aynısı)

In [ ]:
import cv2, matplotlib.pyplot as plt
ornek = sorted((YEREL/'glaucoma').glob('*'))[0]
bgr = cv2.imread(str(ornek))

print(cnn.tahmin(bgr))
harita = cnn.gradcam(bgr)
rgb = cnn.cnn_girdisi_hazirla(bgr)
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(cnn.isi_haritasi_bindir(rgb, harita)); plt.axis('off')
plt.show()

## 6. Çıktıları Drive'a kopyala

In [ ]:
hedef = KAYIT / 'reports'
(hedef / 'figures').mkdir(parents=True, exist_ok=True)
for dosya in ['cnn_egitim_gecmisi.json', 'cnn_test_tahminleri.csv',
              'cnn_kenar_testi.csv', 'cnn_kenar_testi_ozet.csv']:
    shutil.copy(PROJE/'reports'/dosya, hedef/dosya)
for dosya in ['cnn_egitim_grafigi.png', 'gradcam_ornekleri.png']:
    shutil.copy(PROJE/'reports'/'figures'/dosya, hedef/'figures'/dosya)
print('Drive\'a kaydedildi:', hedef, '| model:', MODEL_YOLU)